# Feature Engineering

## project objective

In this notebook, we create new features from the processed AQI dataset. Feature engineering is an important step in the machine learning pipeline, as it enhances the quality of the input data by extracting meaningful information from existing features. These newly created features help machine learning models identify underlying patterns more effectively, ultimately leading to improved prediction accuracy and better overall model performance.

In [1]:
import pandas as pd

In [2]:
df=pd.read_csv('../data/processed/aqi_processed_data.csv')
df.head()

,datetime,AQI,CO,NO,NO2,O3,SO2,PM2_5,PM10,NH3
0,1970-01-01 00:00:01.783731600,3,67.63,0.00,0.68,38.67,0.74,14.64,61.21,0.24
1,1970-01-01 00:00:01.783735200,3,68.75,0.01,0.88,38.99,0.81,14.09,58.51,0.25
2,1970-01-01 00:00:01.783738800,3,70.31,0.06,1.07,40.05,0.90,13.57,55.94,0.23
3,1970-01-01 00:00:01.783742400,3,71.19,0.12,1.03,43.18,0.95,13.26,55.49,0.20
4,1970-01-01 00:00:01.783746000,3,71.06,0.14,0.80,47.91,0.95,12.99,53.78,0.18


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 115 entries, 0 to 114
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   datetime  115 non-null    str    
 1   AQI       115 non-null    int64  
 2   CO        115 non-null    float64
 3   NO        115 non-null    float64
 4   NO2       115 non-null    float64
 5   O3        115 non-null    float64
 6   SO2       115 non-null    float64
 7   PM2_5     115 non-null    float64
 8   PM10      115 non-null    float64
 9   NH3       115 non-null    float64
dtypes: float64(8), int64(1), str(1)
memory usage: 12.4 KB


In [4]:
df["datetime"]= pd.to_datetime(df["datetime"])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 115 entries, 0 to 114
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   datetime  115 non-null    datetime64[ns]
 1   AQI       115 non-null    int64         
 2   CO        115 non-null    float64       
 3   NO        115 non-null    float64       
 4   NO2       115 non-null    float64       
 5   O3        115 non-null    float64       
 6   SO2       115 non-null    float64       
 7   PM2_5     115 non-null    float64       
 8   PM10      115 non-null    float64       
 9   NH3       115 non-null    float64       
dtypes: datetime64[ns](1), float64(8), int64(1)
memory usage: 9.1 KB


## Create Time-Based Features

Time-based features are extracted from the datetime column to help the machine learning model learn seasonal and hourly patterns in air quality.


In [5]:
df["hour"] = df["datetime"].dt.hour
df["day"] = df["datetime"].dt.day
df["month"] = df["datetime"].dt.month
df["weekday"] = df["datetime"].dt.day_name()

df.head()

,datetime,AQI,CO,NO,NO2,O3,SO2,PM2_5,PM10,NH3,hour,day,month,weekday
0,1970-01-01 00:00:01.783731600,3,67.63,0.00,0.68,38.67,0.74,14.64,61.21,0.24,0,1,1,Thursday
1,1970-01-01 00:00:01.783735200,3,68.75,0.01,0.88,38.99,0.81,14.09,58.51,0.25,0,1,1,Thursday
2,1970-01-01 00:00:01.783738800,3,70.31,0.06,1.07,40.05,0.90,13.57,55.94,0.23,0,1,1,Thursday
3,1970-01-01 00:00:01.783742400,3,71.19,0.12,1.03,43.18,0.95,13.26,55.49,0.20,0,1,1,Thursday
4,1970-01-01 00:00:01.783746000,3,71.06,0.14,0.80,47.91,0.95,12.99,53.78,0.18,0,1,1,Thursday


## AQI change rate

A new feature is created to measure how the AQI changes overtime. This helps teh model weather air quality is improving or worsening between consecutives observations.

In [6]:
df["AQI_change"] = df["AQI"].diff() 
df[["datetime", "AQI", "AQI_change"]].head(10)

,datetime,AQI,AQI_change
0,1970-01-01 00:00:01.783731600,3,NaN
1,1970-01-01 00:00:01.783735200,3,0.0
2,1970-01-01 00:00:01.783738800,3,0.0
3,1970-01-01 00:00:01.783742400,3,0.0
4,1970-01-01 00:00:01.783746000,3,0.0
5,1970-01-01 00:00:01.783749600,3,0.0
6,1970-01-01 00:00:01.783753200,2,-1.0
7,1970-01-01 00:00:01.783756800,2,0.0
8,1970-01-01 00:00:01.783760400,2,0.0
9,1970-01-01 00:00:01.783764000,3,1.0


## handle missing values

The first row contains a missing values because there is no previous observation to calculate the change in AQI. We will fill this missing value with 0, indicating no change in AQI for the first observation.

In [7]:
df["AQI_change"]= df["AQI_change"].fillna(0)
df[["AQI", "AQI_change"]].head()

,AQI,AQI_change
0,3,0.0
1,3,0.0
2,3,0.0
3,3,0.0
4,3,0.0


## Create a rolling average feature

A rolling average of AQI is calculated over a specified window size to smooth out short-term fluctuations and highlight longer-term trends in air quality. This feature helps the model capture the overall trend in AQI, which can be useful for predicting future values.

In [ ]:
df["AQI_rolling_avg"] = df["AQI"].rolling(window=3).mean()
df[["AQI", "AQI_rolling_avg"]].head(10)

,AQI,AQI_rolling_avg
0,3,NaN
1,3,NaN
2,3,3.0
3,3,3.0
4,3,3.0
5,3,3.0
6,2,3.0
7,2,2.0
8,2,2.0
9,3,2.0


## Handle missing values

The missing value generated by the rolling values avarage calculation are replaced backward filling to ensure the database is complete for machine learning.

In [10]:
df["AQI_rolling_avg"]=df["AQI_rolling_avg"].bfill()
df[["AQI","AQI_rolling_avg"]].head()

,AQI,AQI_rolling_avg
0,3,3.0
1,3,3.0
2,3,3.0
3,3,3.0
4,3,3.0


## Save feature engineered dataset

The feature engineered dataset is saved for use in exploratory data analysis and machine learning model development.

In [11]:
df.to_csv("../data/processed/AQI_feature_engineered.csv", index=False)
print("feature engineered dataset saved sucessfully.")

feature engineered dataset saved sucessfully.


In [13]:
df.columns

Index(['datetime', 'AQI', 'CO', 'NO', 'NO2', 'O3', 'SO2', 'PM2_5', 'PM10',
       'NH3', 'hour', 'day', 'month', 'weekday', 'AQI_change',
       'AQI_rolling_avg'],
      dtype='str')

In [12]:
df.head(10)

,datetime,AQI,CO,NO,NO2,O3,SO2,PM2_5,PM10,NH3,hour,day,month,weekday,AQI_change,AQI_rolling_avg
0,1970-01-01 00:00:01.783731600,3,67.63,0.00,0.68,38.67,0.74,14.64,61.21,0.24,0,1,1,Thursday,0.0,3.0
1,1970-01-01 00:00:01.783735200,3,68.75,0.01,0.88,38.99,0.81,14.09,58.51,0.25,0,1,1,Thursday,0.0,3.0
2,1970-01-01 00:00:01.783738800,3,70.31,0.06,1.07,40.05,0.90,13.57,55.94,0.23,0,1,1,Thursday,0.0,3.0
3,1970-01-01 00:00:01.783742400,3,71.19,0.12,1.03,43.18,0.95,13.26,55.49,0.20,0,1,1,Thursday,0.0,3.0
4,1970-01-01 00:00:01.783746000,3,71.06,0.14,0.80,47.91,0.95,12.99,53.78,0.18,0,1,1,Thursday,0.0,3.0
5,1970-01-01 00:00:01.783749600,3,70.67,0.13,0.60,52.72,0.93,12.36,50.85,0.16,0,1,1,Thursday,0.0,3.0
6,1970-01-01 00:00:01.783753200,2,71.10,0.11,0.50,56.66,0.88,11.50,48.63,0.14,0,1,1,Thursday,-1.0,3.0
7,1970-01-01 00:00:01.783756800,2,70.89,0.10,0.49,56.53,0.85,11.09,47.48,0.13,0,1,1,Thursday,0.0,2.0
8,1970-01-01 00:00:01.783760400,2,70.18,0.11,0.51,53.66,0.85,11.47,49.35,0.12,0,1,1,Thursday,0.0,2.0
9,1970-01-01 00:00:01.783764000,3,70.24,0.11,0.54,50.72,0.85,11.84,51.28,0.11,0,1,1,Thursday,1.0,2.0


In [14]:
df.shape

(115, 16)